# Détection des Artefacts — Silver Final

Ce notebook détecte toutes les lignes contenant des artefacts dans le dataset Silver.
Les IDs trouvés peuvent être utilisés directement dans Label Studio pour corriger les lignes.

**Artefacts recherchés :**
- `<unk>` / `[unk]` / `unknown`
- `@@` / `@-@`
- `#` (hashtags parasites)
- Espaces doubles
- Lignes tronquées (trop courtes par rapport à la classe)
- Caractères arabes dans `darija_arabizi`
- Caractères latins dans `darija_arabic`
- Caractères arabes dans `english`

In [1]:
import pandas as pd
import re

# ── Charger le dataset ────────────────────────────────────────────────
df = pd.read_csv('./deliverables/gold_final.csv')
print(f'Dataset chargé : {len(df):,} lignes')
print(f'Colonnes : {list(df.columns)}')

Dataset chargé : 1,003 lignes
Colonnes : ['data_id', 'id', 'classe', 'darija_arabic', 'darija_arabizi', 'english', 'modern_standard_arabic', 'status', 'dataset_type']


In [2]:
# ── Patterns de détection ─────────────────────────────────────────────
ARABIC_PAT  = re.compile(r'[\u0600-\u06FF]')
LATIN_PAT   = re.compile(r'[a-zA-Z]')

# Artefacts textuels
UNK_PAT     = re.compile(r'<unk>|\[unk\]|\bunknown\b', re.IGNORECASE)
AROBAS_PAT  = re.compile(r'@@|@-@')
HASH_PAT    = re.compile(r'(?<![a-zA-Z0-9])#(?![\d])')  # # parasite (pas les hashtags légitimes)
DOUBLE_SPACE = re.compile(r'  +')  # 2 espaces ou plus
TRUNCATED_PAT = re.compile(r'\.\.\.\s*$|–\s*$|—\s*$')  # phrase coupée

def detect_artefacts(text):
    """Retourne la liste des artefacts trouvés dans un texte."""
    text = str(text) if pd.notna(text) else ''
    found = []
    if UNK_PAT.search(text):      found.append('<unk>/unknown')
    if AROBAS_PAT.search(text):   found.append('@@/@-@')
    if HASH_PAT.search(text):     found.append('#parasite')
    if DOUBLE_SPACE.search(text): found.append('espaces doubles')
    if TRUNCATED_PAT.search(text):found.append('phrase tronquée')
    return found

print('Patterns définis.')

Patterns définis.


In [3]:
# ── Détection sur toutes les colonnes ─────────────────────────────────
COLS_TO_CHECK = ['darija_arabic', 'darija_arabizi', 'english', 'modern_standard_arabic']

results = []

for _, row in df.iterrows():
    row_issues = []

    for col in COLS_TO_CHECK:
        text = str(row.get(col, '') or '')
        artefacts = detect_artefacts(text)
        for a in artefacts:
            row_issues.append({
                'data_id' : row.get('data_id', ''),
                'id'      : row.get('id', ''),
                'classe'  : row.get('classe', ''),
                'colonne' : col,
                'artefact': a,
                'valeur'  : text[:120] + ('...' if len(text) > 120 else ''),
            })

    # Script errors
    darija_a = str(row.get('darija_arabic',  '') or '')
    darija_z = str(row.get('darija_arabizi', '') or '')
    english  = str(row.get('english',        '') or '')

    if ARABIC_PAT.search(darija_z) and len(darija_z) > 0:
        row_issues.append({
            'data_id' : row.get('data_id', ''),
            'id'      : row.get('id', ''),
            'classe'  : row.get('classe', ''),
            'colonne' : 'darija_arabizi',
            'artefact': 'ARABE dans Arabizi',
            'valeur'  : darija_z[:120],
        })

    if darija_a and not ARABIC_PAT.search(darija_a):
        row_issues.append({
            'data_id' : row.get('data_id', ''),
            'id'      : row.get('id', ''),
            'classe'  : row.get('classe', ''),
            'colonne' : 'darija_arabic',
            'artefact': 'LATIN dans Darija Arabic',
            'valeur'  : darija_a[:120],
        })

    if ARABIC_PAT.search(english):
        row_issues.append({
            'data_id' : row.get('data_id', ''),
            'id'      : row.get('id', ''),
            'classe'  : row.get('classe', ''),
            'colonne' : 'english',
            'artefact': 'ARABE dans English',
            'valeur'  : english[:120],
        })

    results.extend(row_issues)

df_issues = pd.DataFrame(results)
print(f'\nTotal anomalies détectées : {len(df_issues)}')
print(f'Lignes uniques concernées : {df_issues["id"].nunique() if len(df_issues) > 0 else 0}')


Total anomalies détectées : 58
Lignes uniques concernées : 48


In [4]:
# ── Résumé par type d'artefact ────────────────────────────────────────
if len(df_issues) > 0:
    print('\n=== RÉSUMÉ PAR TYPE D\'ARTEFACT ===')
    summary = df_issues.groupby('artefact').agg(
        nb_lignes=('id', 'nunique'),
        nb_occurrences=('id', 'count')
    ).sort_values('nb_lignes', ascending=False)
    print(summary.to_string())

    print('\n=== RÉSUMÉ PAR COLONNE ===')
    print(df_issues.groupby('colonne')['id'].nunique().sort_values(ascending=False).to_string())
else:
    print('Aucune anomalie détectée — dataset propre !')


=== RÉSUMÉ PAR TYPE D'ARTEFACT ===
                          nb_lignes  nb_occurrences
artefact                                           
espaces doubles                  40              44
<unk>/unknown                     3               3
LATIN dans Darija Arabic          3               3
phrase tronquée                   2               8

=== RÉSUMÉ PAR COLONNE ===
colonne
darija_arabic             25
modern_standard_arabic    14
english                   13
darija_arabizi             6


In [5]:
# ── Détail : <unk> / unknown ──────────────────────────────────────────
unk_issues = df_issues[df_issues['artefact'] == '<unk>/unknown']
print(f'Lignes avec <unk>/unknown : {len(unk_issues)}')
if len(unk_issues) > 0:
    print('\nIDs à corriger dans Label Studio :')
    for _, row in unk_issues.iterrows():
        print(f"  → ID: {row['id']} | Colonne: {row['colonne']}")
        print(f"    Valeur: {row['valeur']}")
        print()

Lignes avec <unk>/unknown : 3

IDs à corriger dans Label Studio :
  → ID: WIKI_C_06338 | Colonne: english
    Valeur: Given the planet 's high mass, the planet is a gas giant with no solid surface. Since the planet has only been detected ...

  → ID: WIKI_D_07842 | Colonne: english
    Valeur: Introduced by the second International Opium Convention , the Permanent Central Opium Board had to supervise the statist...

  → ID: WIKI_C_03862 | Colonne: english
    Valeur: Twenty three species of butterflies and 105 species of moth enjoy the floral diversity of the site . The holly blue ( <u...



In [6]:
# ── Détail : @@ / @-@ ────────────────────────────────────────────────
arobas_issues = df_issues[df_issues['artefact'] == '@@/@-@']
print(f'Lignes avec @@ / @-@ : {len(arobas_issues)}')
if len(arobas_issues) > 0:
    print('\nIDs à corriger dans Label Studio :')
    for _, row in arobas_issues.iterrows():
        print(f"  → ID: {row['id']} | Colonne: {row['colonne']}")
        print(f"    Valeur: {row['valeur']}")
        print()

Lignes avec @@ / @-@ : 0


In [7]:
# ── Détail : erreurs de script ────────────────────────────────────────
script_issues = df_issues[df_issues['artefact'].isin([
    'ARABE dans Arabizi', 'LATIN dans Darija Arabic', 'ARABE dans English'
])]
print(f'Lignes avec erreurs de script : {len(script_issues)}')
if len(script_issues) > 0:
    print('\nIDs à corriger dans Label Studio :')
    for _, row in script_issues.iterrows():
        print(f"  → ID: {row['id']} | Problème: {row['artefact']}")
        print(f"    Valeur: {row['valeur']}")
        print()

Lignes avec erreurs de script : 3

IDs à corriger dans Label Studio :
  → ID: DATA_21312 | Problème: LATIN dans Darija Arabic
    Valeur: 1964 – 65 , 1973 – 74 , 1985 – 86 , 1988 – 89 , 1991 – 92 , 2000 – 01 , 2005 – 06

  → ID: WIKI_D_05207 | Problème: LATIN dans Darija Arabic
    Valeur: w 7ed l-EP tab3 tsemma Trouble – Norwegian Live EP tlqoh f ssyf dyal 2001, fih 5 dyal l-morso tsjlo live men tarf l-fraq

  → ID: DATA_14117 | Problème: LATIN dans Darija Arabic
    Valeur: C (s) + H2O (g) -> CO (g) + H2 (g)



In [8]:
# ── Export des résultats ──────────────────────────────────────────────
if len(df_issues) > 0:
    # Fichier complet de toutes les anomalies
    df_issues.to_csv('artefacts_detected.csv', index=False, encoding='utf-8-sig')
    print('artefacts_detected.csv exporté')

    # Liste unique des IDs à corriger (pour copier-coller dans Label Studio)
    ids_to_fix = df_issues['id'].unique().tolist()
    ids_df = pd.DataFrame({'id': ids_to_fix})
    ids_df.to_csv('ids_to_fix.csv', index=False)
    print(f'ids_to_fix.csv exporté : {len(ids_to_fix)} IDs uniques à corriger')

    print(f'\n=== LISTE DES IDs ===')
    for id_ in ids_to_fix:
        print(f'  {id_}')
else:
    print('Rien à exporter — dataset propre !')

artefacts_detected.csv exporté
ids_to_fix.csv exporté : 48 IDs uniques à corriger

=== LISTE DES IDs ===
  DATA_16419
  DATA_7330
  WIKI_D_08734
  WIKI_D_03779
  WIKI_C_05258
  WIKI_C_04276
  WIKI_C_06338
  DATA_22840
  DATA_17817
  WIKI_D_10295
  WIKI_D_09946
  DATA_21312
  DATA_22175
  DATA_13680
  WIKI_C_06735
  WIKI_D_08618
  WIKI_D_06462
  WIKI_C_01545
  DATA_21405
  DATA_2859
  WIKI_D_08448
  DATA_12755
  DATA_19909
  WIKI_C_04948
  WIKI_D_05629
  DATA_18235
  WIKI_C_11357
  DATA_16513
  WIKI_C_03347
  WIKI_D_02949
  WIKI_D_02581
  DATA_17903
  WIKI_D_06437
  WIKI_C_06604
  WIKI_D_07842
  WIKI_D_04431
  WIKI_D_05207
  WIKI_C_08200
  WIKI_D_00978
  DATA_18716
  DATA_22964
  WIKI_D_09959
  DATA_0598
  DATA_10582
  DATA_14117
  WIKI_C_03862
  WIKI_C_09285
  WIKI_C_01147


In [9]:
# ── Nettoyage automatique des artefacts SIMPLES ───────────────────────
# (espaces doubles, @@ @-@ seulement — le reste nécessite correction manuelle)

df_clean = df.copy()

def auto_clean(text):
    if pd.isna(text): return text
    text = str(text)
    text = AROBAS_PAT.sub('', text)       # supprimer @@ et @-@
    text = DOUBLE_SPACE.sub(' ', text)    # normaliser espaces doubles
    text = text.strip()
    return text

for col in COLS_TO_CHECK:
    df_clean[col] = df_clean[col].apply(auto_clean)

df_clean.to_csv('silver_final_cleaned.csv', index=False, encoding='utf-8-sig')
print('silver_final_cleaned.csv exporté')
print('NOTE : <unk>, erreurs de script et phrases tronquées nécessitent correction manuelle dans Label Studio.')

silver_final_cleaned.csv exporté
NOTE : <unk>, erreurs de script et phrases tronquées nécessitent correction manuelle dans Label Studio.
